# 02 - Análise de Qualidade dos Dados - Camada Bronze

## Objetivo

Este notebook realiza o diagnóstico de qualidade dos dados armazenados
na camada Bronze do Lakehouse.

A análise considera as seguintes dimensões:

- Completude: identificação de valores nulos ou ausentes;
- Unicidade: identificação de registros duplicados e validação das chaves;
- Consistência: verificação de padrões, categorias e formatos esperados;
- Acurácia/Plausibilidade: identificação de valores incompatíveis com o contexto;
- Outliers: identificação de valores extremos potencialmente relevantes.

Nenhuma transformação corretiva será realizada nesta etapa.
Os problemas identificados serão utilizados como entrada para a construção
da camada Silver.


In [0]:
from pyspark.sql import functions as F

CATALOG = "datalake_mvp"
SCHEMA_BRONZE = "mvp_bronze"

tabelas = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "product_category_translation"
]

In [0]:
inventario = []

for tabela in tabelas:

    df = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{tabela}")

    inventario.append(
        (
            tabela,
            df.count(),
            len(df.columns)
        )
    )

df_inventario = spark.createDataFrame(
    inventario,
    ["tabela", "quantidade_registros", "quantidade_colunas"]
)

display(df_inventario.orderBy("tabela"))

tabela,quantidade_registros,quantidade_colunas
customers,99441,7
geolocation,1000163,7
order_items,112650,9
order_payments,103886,7
order_reviews,99224,9
orders,99441,10
product_category_translation,71,4
products,32951,11
sellers,3095,6


In [0]:
for tabela in tabelas:

    print("=" * 80)
    print(f"TABELA: {tabela}")
    print("=" * 80)

    df = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{tabela}")

    df.printSchema()

TABELA: customers
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

TABELA: geolocation
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

TABELA: order_items
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: 

In [0]:
resultado_nulos = []

for tabela in tabelas:

    df = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{tabela}")

    total_registros = df.count()

    for coluna in df.columns:

        total_nulos = (
            df
            .filter(F.col(coluna).isNull())
            .count()
        )

        percentual_nulos = (
            (total_nulos / total_registros) * 100
            if total_registros > 0
            else 0
        )

        resultado_nulos.append(
            (
                tabela,
                coluna,
                total_registros,
                total_nulos,
                round(percentual_nulos, 2)
            )
        )

In [0]:
df_completude = spark.createDataFrame(
    resultado_nulos,
    [
        "tabela",
        "coluna",
        "total_registros",
        "total_nulos",
        "percentual_nulos"
    ]
)

display(
    df_completude
    .filter(F.col("total_nulos") > 0)
    .orderBy(
        F.desc("percentual_nulos"),
        "tabela",
        "coluna"
    )
)

tabela,coluna,total_registros,total_nulos,percentual_nulos
order_reviews,review_comment_title,99224,87656,88.34
order_reviews,review_comment_message,99224,58247,58.7
orders,order_delivered_customer_date,99441,2965,2.98
products,product_category_name,32951,610,1.85
products,product_description_lenght,32951,610,1.85
products,product_name_lenght,32951,610,1.85
products,product_photos_qty,32951,610,1.85
orders,order_delivered_carrier_date,99441,1783,1.79
orders,order_approved_at,99441,160,0.16
products,product_height_cm,32951,2,0.01


In [0]:
resultado_vazios = []

for tabela in tabelas:

    df = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{tabela}")

    total_registros = df.count()

    colunas_string = [
        campo.name
        for campo in df.schema.fields
        if campo.dataType.simpleString() == "string"
    ]

    for coluna in colunas_string:

        total_vazios = (
            df
            .filter(
                F.trim(F.col(coluna)) == ""
            )
            .count()
        )

        if total_vazios > 0:

            resultado_vazios.append(
                (
                    tabela,
                    coluna,
                    total_vazios,
                    round((total_vazios / total_registros) * 100, 2)
                )
            )

In [0]:
if resultado_vazios:

    df_vazios = spark.createDataFrame(
        resultado_vazios,
        [
            "tabela",
            "coluna",
            "total_vazios",
            "percentual_vazios"
        ]
    )

    display(df_vazios)

else:

    print("Nenhum campo textual vazio foi identificado.")

tabela,coluna,total_vazios,percentual_vazios
order_reviews,review_comment_title,2,0.0
order_reviews,review_comment_message,9,0.01


## Análise de Unicidade

Nesta etapa são avaliadas as chaves naturais ou compostas esperadas para
cada conjunto de dados.

A análise considera a granularidade de cada tabela, evitando assumir
unicidade para atributos que legitimamente podem se repetir, como
`order_id` nas tabelas de itens e pagamentos.

In [0]:
chaves = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "order_items": ["order_id", "order_item_id"],
    "product_category_translation": ["product_category_name"]
}

resultado_unicidade = []

for tabela, chave in chaves.items():

    df = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{tabela}")

    total = df.count()

    total_chaves_distintas = (
        df
        .select(*chave)
        .distinct()
        .count()
    )

    duplicados = total - total_chaves_distintas

    resultado_unicidade.append(
        (
            tabela,
            " + ".join(chave),
            total,
            total_chaves_distintas,
            duplicados
        )
    )

df_unicidade = spark.createDataFrame(
    resultado_unicidade,
    [
        "tabela",
        "chave_avaliada",
        "total_registros",
        "chaves_distintas",
        "registros_duplicados"
    ]
)

display(df_unicidade.orderBy("tabela"))

tabela,chave_avaliada,total_registros,chaves_distintas,registros_duplicados
customers,customer_id,99441,99441,0
order_items,order_id + order_item_id,112650,112650,0
orders,order_id,99441,99441,0
product_category_translation,product_category_name,71,71,0
products,product_id,32951,32951,0
sellers,seller_id,3095,3095,0


In [0]:
df_reviews = spark.table(
    "datalake_mvp.mvp_bronze.order_reviews"
)

display(
    df_reviews
    .groupBy("review_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

review_id,count
f4bb9d6dd4fb6dcc2298f0e7b17b8e1e,3
08528f70f579f0c830189efc523d2182,3
e44840754f12fad2b8646712121b349a,3
69a1068c3128a14994e3e422e4539e04,3
3415c9f764e478409e8e0660ae816dd2,3
70509c441d994fa03d6c1457930c9024,3
0c76e7a547a531e7bf9f0b99cba071c1,3
832acec9bbf4efe65c3fb6423d8b4ed7,3
38821b5c496b678cf91acc34892805ad,3
7b606b0d57b078384f0b58eac1d41d78,3


In [0]:
display(
    df_reviews
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

order_id,count
c88b1d1b157a9999ce368f218a407141,3
03c939fd7fd3b38f8485a0f95798f1f6,3
8e17072ec97ce29f0e1f111e598b0c85,3
df56136b8031ecd28e200bb18e6ddb2e,3
715b8576b74d53796bcbb107a201eb55,2
dbf76ef1323584fa387945f83b324534,2
7139a3b215ea730e2db624cc99820578,2
99ced390cb3920dd78d289dcddfc6ce4,2
6540a36b3dd6a4a0922d5f57d493d9d6,2
8f5fac100b291e3c7c7c34ca50001b5a,2


In [0]:
df_payments = spark.table(
    "datalake_mvp.mvp_bronze.order_payments"
)

display(
    df_payments
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

order_id,count
fa65dad1b0e818e3ccc5cb0e39231352,29
ccf804e764ed5650cd8759557269dc13,26
285c2e15bebd4ac83635ccc563dc71f4,22
895ab968e7bb0d5659d16cd74cd1650c,21
ee9ca989fc93ba09a6eddc250ce01742,19
fedcd9f7ccdc8cba3a18defedd1a5547,19
4bfcba9e084f46c8e3cb49b0fa6e6159,15
21577126c19bf11a0b91592e5844ba78,15
3c58bffb70dcf45f12bdf66a3c215905,14
4689b1816de42507a7d63a4617383c59,14


In [0]:
df_geo = spark.table(
    "datalake_mvp.mvp_bronze.geolocation"
)

display(
    df_geo
    .groupBy("geolocation_zip_code_prefix")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

geolocation_zip_code_prefix,count
24220,1146
24230,1102
38400,965
35500,907
11680,879
22631,832
30140,810
11740,788
38408,773
28970,743


In [0]:
display(
    df_reviews
    .filter(
        F.col("order_id").isNull() |
        (F.trim(F.col("order_id")) == "")
    )
)

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,_ingestion_timestamp,_source_file


In [0]:
caminho_reviews = (
    "/Volumes/datalake_mvp/mvp_bronze/raw_files/"
    "olist_order_reviews_dataset.csv"
)

df_reviews_teste = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(caminho_reviews)
)

print(f"Registros atuais Bronze: {df_reviews.count()}")
print(f"Registros com leitura multiline: {df_reviews_teste.count()}")

Registros atuais Bronze: 99224
Registros com leitura multiline: 99224


In [0]:
display(
    df_reviews_teste.select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp"
    )
)

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13T00:00:00.000Z,2018-04-16T00:39:37.000Z
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16T00:00:00.000Z,2017-07-18T19:30:34.000Z
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14T00:00:00.000Z,2018-08-14T21:36:06.000Z
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17T00:00:00.000Z,2017-05-18T12:05:37.000Z
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22T00:00:00.000Z,2018-05-23T16:45:47.000Z


In [0]:
display(
    df_reviews_teste.select([
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in df_reviews_teste.columns
    ])
)

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,0,0,87656,58247,0,0



### Achado de Qualidade - Arquivo de Avaliações

Durante a análise de qualidade da tabela `order_reviews`, foram
identificados valores textuais incompatíveis com o domínio esperado
do atributo `review_id`, além de quantidade inesperada de valores
nulos em atributos estruturais como `order_id` e `review_score`.

A inspeção dos registros indicou possível desalinhamento das colunas
durante a leitura do arquivo CSV, potencialmente causado pela presença
de campos textuais com delimitadores e/ou quebras de linha.

Por esse motivo, o arquivo original foi reavaliado antes da aplicação
de qualquer tratamento de valores nulos, evitando que um problema de
ingestão fosse incorretamente classificado como ausência de dados.

## Conclusões - Completude e Unicidade

A análise de completude identificou que os principais valores ausentes da
tabela de avaliações estão concentrados nos campos textuais opcionais
`review_comment_title` e `review_comment_message`. Esses valores não serão
imputados, pois a ausência de comentário não representa necessariamente
um erro de qualidade.

Durante a análise inicial foi identificado um problema de parsing no arquivo
de avaliações, provocado pela presença de campos textuais multilinha. Após
a adequação da ingestão, a tabela passou de 104.162 registros interpretados
incorretamente para 99.224 registros, eliminando valores nulos artificiais
em atributos estruturais.

As principais entidades apresentaram unicidade adequada nas chaves avaliadas.
As tabelas de avaliações, pagamentos e geolocalização possuem granularidades
específicas e, portanto, não devem ser avaliadas utilizando apenas `order_id`
ou prefixo de CEP como chave única.

Esses resultados serão considerados na definição das regras de tratamento
e modelagem das camadas Silver e Gold.

## Análise de Consistência e Plausibilidade

Nesta etapa são avaliadas regras de domínio e de negócio com o objetivo
de identificar valores estruturalmente válidos, porém incompatíveis com
o comportamento esperado dos dados.

Serão analisados status de pedidos, notas de avaliações, valores monetários,
quantidades, atributos físicos dos produtos e coerência temporal.

In [0]:
# Status existentes nos pedidos
df_orders = spark.table("datalake_mvp.mvp_bronze.orders")

display(
    df_orders
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

order_status,count
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [0]:
display(
    df_reviews
    .groupBy("review_score")
    .count()
    .orderBy("review_score")
)

review_score,count
1,11424
2,3151
3,8179
4,19142
5,57328


In [0]:
display(
    df_payments
    .groupBy("payment_type")
    .count()
    .orderBy(F.desc("count"))
)

payment_type,count
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


In [0]:
display(
    df_payments.filter(
        (F.col("payment_value") < 0) |
        (F.col("payment_installments") < 0)
    )
)

order_id,payment_sequential,payment_type,payment_installments,payment_value,_ingestion_timestamp,_source_file


In [0]:
df_items = spark.table(
    "datalake_mvp.mvp_bronze.order_items"
)

display(
    df_items.filter(
        (F.col("price") < 0) |
        (F.col("freight_value") < 0)
    )
)

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,_ingestion_timestamp,_source_file


In [0]:
df_products = spark.table(
    "datalake_mvp.mvp_bronze.products"
)

display(
    df_products.filter(
        (F.col("product_weight_g") <= 0) |
        (F.col("product_length_cm") <= 0) |
        (F.col("product_height_cm") <= 0) |
        (F.col("product_width_cm") <= 0)
    )
)

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_ingestion_timestamp,_source_file
81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51,529,1,0,30,25,30,2026-09-23T00:58:34.787Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_products_dataset.csv
8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48,528,1,0,30,25,30,2026-09-23T00:58:34.787Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_products_dataset.csv
36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53,528,1,0,30,25,30,2026-09-23T00:58:34.787Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_products_dataset.csv
e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53,528,1,0,30,25,30,2026-09-23T00:58:34.787Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_products_dataset.csv


In [0]:
display(
    df_orders.filter(
        (
            F.col("order_approved_at").isNotNull() &
            (F.col("order_approved_at") < F.col("order_purchase_timestamp"))
        )
        |
        (
            F.col("order_delivered_carrier_date").isNotNull() &
            F.col("order_approved_at").isNotNull() &
            (F.col("order_delivered_carrier_date") < F.col("order_approved_at"))
        )
        |
        (
            F.col("order_delivered_customer_date").isNotNull() &
            F.col("order_delivered_carrier_date").isNotNull() &
            (F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"))
        )
    )
)

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_ingestion_timestamp,_source_file
dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07T19:03:12.000Z,2018-06-12T23:31:02.000Z,2018-06-11T14:54:00.000Z,2018-06-21T15:34:32.000Z,2018-07-04T00:00:00.000Z,2026-09-23T00:58:31.593Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_orders_dataset.csv
688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22T08:48:13.000Z,2018-04-24T18:25:22.000Z,2018-04-23T19:19:14.000Z,2018-04-24T19:31:58.000Z,2018-05-15T00:00:00.000Z,2026-09-23T00:58:31.593Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_orders_dataset.csv
58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21T12:49:32.000Z,2018-07-26T23:31:53.000Z,2018-07-24T12:57:00.000Z,2018-07-25T23:58:19.000Z,2018-07-31T00:00:00.000Z,2026-09-23T00:58:31.593Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_orders_dataset.csv
412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22T22:30:05.000Z,2018-07-23T12:31:53.000Z,2018-07-23T12:24:00.000Z,2018-07-24T19:26:42.000Z,2018-07-31T00:00:00.000Z,2026-09-23T00:58:31.593Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_orders_dataset.csv
56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22T13:04:47.000Z,2018-07-27T23:31:09.000Z,2018-07-24T14:03:00.000Z,2018-07-28T00:05:39.000Z,2018-08-06T00:00:00.000Z,2026-09-23T00:58:31.593Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_orders_dataset.csv
32e4fa9bb468884309b58b9348de70c3,e54367d4b00c5cb76d2dfe71b9bdb89c,delivered,2018-07-04T16:49:21.000Z,2018-07-05T16:33:06.000Z,2018-07-05T14:50:00.000Z,2018-07-07T14:41:18.000Z,2018-07-23T00:00:00.000Z,2026-09-23T00:58:31.593Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_orders_dataset.csv
4df92d82d79c3b52c7138679fa9b07fc,ba0660bf3fffe505ee892e153a2fbd49,delivered,2018-07-24T11:32:11.000Z,2018-07-29T23:30:52.000Z,2018-07-26T14:46:00.000Z,2018-07-27T18:55:57.000Z,2018-08-06T00:00:00.000Z,2026-09-23T00:58:31.593Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_orders_dataset.csv
16e38caa92e342c7780f68832f832d4d,d29935fdaf4a76f34653da3def4f2a24,delivered,2018-05-07T01:09:09.000Z,2018-05-07T16:52:39.000Z,2018-05-07T15:09:00.000Z,2018-05-24T00:31:18.000Z,2018-06-07T00:00:00.000Z,2026-09-23T00:58:31.593Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_orders_dataset.csv
b9afddbdcfadc9a87b41a83271c3e888,85c6af75161b8b2b1af98e82b5a6a5a5,delivered,2018-08-16T13:50:48.000Z,2018-08-16T14:05:13.000Z,2018-08-16T13:27:00.000Z,2018-08-24T14:58:37.000Z,2018-09-04T00:00:00.000Z,2026-09-23T00:58:31.593Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_orders_dataset.csv
6051e6d3da9a50b7325cbe9c81025062,c25122206bfcb3ffa950706e1cb10f2b,delivered,2018-07-03T23:40:16.000Z,2018-07-05T16:31:26.000Z,2018-07-04T12:14:00.000Z,2018-07-05T22:52:28.000Z,2018-07-19T00:00:00.000Z,2026-09-23T00:58:31.593Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_orders_dataset.csv


In [0]:
display(
    df_orders
    .filter(
        F.col("order_delivered_customer_date").isNotNull() &
        F.col("order_estimated_delivery_date").isNotNull() &
        (
            F.col("order_delivered_customer_date") >
            F.col("order_estimated_delivery_date")
        )
    )
    .select(
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
)

order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
203096f03d82e0dffbc41ebc2e2bcfb7,2017-09-18T14:31:30.000Z,2017-10-09T22:23:46.000Z,2017-09-28T00:00:00.000Z
fbf9ac61453ac646ce8ad9783d7d0af6,2018-02-20T23:46:53.000Z,2018-03-21T22:03:54.000Z,2018-03-12T00:00:00.000Z
8563039e855156e48fccee4d611a3196,2018-02-17T15:59:46.000Z,2018-03-20T00:59:25.000Z,2018-03-20T00:00:00.000Z
6ea2f835b4556291ffdc53fa0b3b95e8,2017-11-24T21:27:48.000Z,2017-12-28T18:59:23.000Z,2017-12-21T00:00:00.000Z
66e4624ae69e7dc89bd50222b59f581f,2018-03-09T14:50:15.000Z,2018-04-03T13:28:46.000Z,2018-04-02T00:00:00.000Z
a685d016c8a26f71a0bb67821070e398,2017-03-13T18:14:36.000Z,2017-04-06T13:37:16.000Z,2017-03-30T00:00:00.000Z
6a0a8bfbbe700284feb0845d95e0867f,2017-11-22T11:32:22.000Z,2017-12-28T19:43:00.000Z,2017-12-11T00:00:00.000Z
a5474c0071dd5d1074e12d417078bbd0,2018-07-30T22:41:44.000Z,2018-08-03T19:28:47.000Z,2018-08-02T00:00:00.000Z
9d531c565e28c3e0d756192f84d8731f,2017-11-28T21:00:44.000Z,2018-01-23T21:38:52.000Z,2017-12-22T00:00:00.000Z
8fc207e94fa91a7649c5a5dab690272a,2017-11-26T17:49:46.000Z,2018-01-20T13:42:22.000Z,2017-12-19T00:00:00.000Z


In [0]:
regras_temporais = [
    (
        "aprovacao_antes_compra",
        F.col("order_approved_at").isNotNull() &
        (F.col("order_approved_at") < F.col("order_purchase_timestamp"))
    ),
    (
        "envio_antes_aprovacao",
        F.col("order_delivered_carrier_date").isNotNull() &
        F.col("order_approved_at").isNotNull() &
        (F.col("order_delivered_carrier_date") < F.col("order_approved_at"))
    ),
    (
        "entrega_antes_envio",
        F.col("order_delivered_customer_date").isNotNull() &
        F.col("order_delivered_carrier_date").isNotNull() &
        (F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"))
    )
]

resultado_temporal = []

for regra, condicao in regras_temporais:

    quantidade = df_orders.filter(condicao).count()

    resultado_temporal.append(
        (regra, quantidade)
    )

df_regras_temporais = spark.createDataFrame(
    resultado_temporal,
    ["regra", "quantidade_inconsistencias"]
)

display(df_regras_temporais)

regra,quantidade_inconsistencias
aprovacao_antes_compra,0
envio_antes_aprovacao,1359
entrega_antes_envio,23


In [0]:
display(
    df_products
    .filter(F.col("product_weight_g") <= 0)
    .select(
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    )
)

product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,0,30,25,30
8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,0,30,25,30
36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,0,30,25,30
e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,0,30,25,30


In [0]:
display(
    df_payments
    .filter(F.col("payment_type") == "not_defined")
)

order_id,payment_sequential,payment_type,payment_installments,payment_value,_ingestion_timestamp,_source_file
4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,2026-09-23T00:58:25.269Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_order_payments_dataset.csv
00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,2026-09-23T00:58:25.269Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_order_payments_dataset.csv
c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0,2026-09-23T00:58:25.269Z,dbfs:/Volumes/datalake_mvp/mvp_bronze/raw_files/olist_order_payments_dataset.csv


## Análise de Outliers

Para identificação de valores extremos em variáveis numéricas foi utilizado
o método do Intervalo Interquartil (IQR).

Um valor é classificado como potencial outlier quando se encontra abaixo de
Q1 - 1,5 × IQR ou acima de Q3 + 1,5 × IQR.

A identificação de um outlier não implica sua remoção automática. Valores
extremos podem representar transações legítimas e relevantes para o negócio.
Os resultados serão avaliados antes da definição de qualquer tratamento.

In [0]:
def analisar_outliers_iqr(df, coluna):

    q1, q3 = df.approxQuantile(
        coluna,
        [0.25, 0.75],
        0.01
    )

    iqr = q3 - q1

    limite_inferior = q1 - (1.5 * iqr)
    limite_superior = q3 + (1.5 * iqr)

    outliers = df.filter(
        (F.col(coluna) < limite_inferior) |
        (F.col(coluna) > limite_superior)
    )

    return (
        q1,
        q3,
        limite_inferior,
        limite_superior,
        outliers.count()
    )

In [0]:
analises_outliers = [
    ("order_items", df_items, "price"),
    ("order_items", df_items, "freight_value"),
    ("order_payments", df_payments, "payment_value"),
    ("products", df_products, "product_weight_g")
]

resultado_outliers = []

for tabela, df, coluna in analises_outliers:

    q1, q3, li, ls, quantidade = analisar_outliers_iqr(
        df,
        coluna
    )

    resultado_outliers.append(
        (
            tabela,
            coluna,
            float(q1),
            float(q3),
            float(li),
            float(ls),
            quantidade
        )
    )

df_outliers = spark.createDataFrame(
    resultado_outliers,
    [
        "tabela",
        "coluna",
        "q1",
        "q3",
        "limite_inferior",
        "limite_superior",
        "quantidade_outliers"
    ]
)

display(df_outliers)

tabela,coluna,q1,q3,limite_inferior,limite_superior,quantidade_outliers
order_items,price,39.9,129.99,-95.23499999999999,265.125,8896
order_items,freight_value,13.0,20.93,1.1050000000000004,32.825,12350
order_payments,payment_value,56.39,167.99,-111.01,335.39,8396
products,product_weight_g,300.0,1800.0,-1950.0,4050.0,4724


# Conclusões da Análise de Qualidade

A análise da camada Bronze permitiu avaliar as dimensões de completude,
unicidade, consistência, plausibilidade e presença de valores extremos.

## Principais achados

**Completude**
- Os principais valores ausentes estão concentrados em atributos opcionais,
  especialmente os campos textuais das avaliações.
- Foi identificado e corrigido durante a ingestão um problema de parsing
  no arquivo de avaliações causado por campos textuais multilinha.

**Unicidade**
- As principais entidades apresentaram unicidade adequada nas chaves avaliadas.
- As tabelas de avaliações, pagamentos e geolocalização possuem granularidades
  específicas e não devem ser avaliadas utilizando apenas `order_id` ou
  prefixo de CEP como chave única.

**Consistência**
- Os status de pedidos apresentaram categorias consistentes.
- As avaliações respeitam o domínio esperado de notas entre 1 e 5.
- Foram identificados três pagamentos classificados como `not_defined`,
  todos com valor igual a zero.

**Plausibilidade**
- Não foram identificados preços, fretes ou valores de pagamento negativos.
- Foram identificados quatro produtos com peso igual a zero, valor
  fisicamente implausível para produtos cadastrados com dimensões válidas.
- Foram identificados 1.359 registros em que a data de envio à transportadora
  antecede a data de aprovação e 23 registros em que a entrega ao cliente
  antecede a data de envio registrada.

**Outliers**
- O método IQR identificou potenciais valores extremos em preço, frete,
  valor de pagamento e peso dos produtos.
- Os outliers não serão removidos automaticamente, pois valores extremos
  podem representar transações legítimas e relevantes para a análise.

## Estratégia para a camada Silver

Com base no diagnóstico, serão aplicadas transformações de padronização,
tratamento e enriquecimento sem eliminar informações potencialmente válidas.

Entre as principais ações previstas estão:

- padronização de valores categóricos;
- conversão de pesos iguais a zero para nulo;
- criação de indicadores de inconsistência temporal;
- manutenção dos outliers identificados;
- tratamento adequado das diferentes granularidades;
- padronização de tipos e nomes de atributos;
- preservação da rastreabilidade dos dados.